# 01 — ACSC Projection Demo
**Abstract.** Demonstrates the ACSC projection map Φ(E): maps elliptic-curve invariants to a 3D symbolic manifold. Loads local Cremona/LMFDB data if available; otherwise generates synthetic invariants. Visualizes the projection, saves points and a scatter figure, and optionally computes a small persistence barcode.


In [ ]:
import sys, os
sys.path.append('src')
import platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(42)
print("Python:", platform.python_version())
print("NumPy:", np.__version__, "Pandas:", pd.__version__)


In [ ]:
os.makedirs('results', exist_ok=True)


In [ ]:
# Try local arithmetic datasets
data_path = None
for p in ['cremona_from_raw_invariants.csv', 'lmfdb_raw_parsed.csv']:
    if os.path.exists(p):
        data_path = p
        break

if data_path:
    df = pd.read_csv(data_path)
    print(f"Loaded arithmetic data from: {data_path}")
    # Attempt to find columns; adapt if necessary
    cols = df.columns.str.lower()
    def safe_col(names):
        for n in names:
            if n in cols:
                return df[df.columns[cols.tolist().index(n)]]
        return None
    # Try common names
    rank = safe_col(['rank','curve_rank'])
    regulator = safe_col(['regulator','reg'])
    conductor = safe_col(['conductor','cond'])
    if rank is None or regulator is None or conductor is None:
        print("Arithmetic file found but required columns missing. Falling back to synthetic.")
        data_path = None

if not data_path:
    print("No local arithmetic data found. Generating synthetic elliptic-curve invariants.")
    n = 300
    df = pd.DataFrame({
        'rank': np.random.randint(0, 4, size=n),
        'regulator': np.random.exponential(scale=1.0, size=n),
        'conductor': np.random.lognormal(mean=5, sigma=1.0, size=n)
    })


In [ ]:
# Use src.projection if available, else fallback
try:
    from src.projection.projection import project_invariants
    print("Using src.projection.project_invariants")
    points = np.vstack(df.apply(lambda r: project_invariants(r), axis=1).to_list())
except Exception:
    print("Using local fallback projection.")
    def fallback_project(row):
        return np.array([row['rank'],
                         np.log10(row['conductor'] + 1e-12),
                         row['regulator']])
    points = np.vstack(df.apply(fallback_project, axis=1).to_list())

df['x'], df['y'], df['z'] = points[:,0], points[:,1], points[:,2]
df.to_csv('results/projection_points.csv', index=False)
print("Saved results/projection_points.csv")


In [ ]:
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(df['x'], df['y'], df['z'], c=df['rank'], s=20 + 30*df['regulator'], cmap='viridis', alpha=0.8)
ax.set_xlabel('Rank'); ax.set_ylabel('log10(Conductor)'); ax.set_zlabel('Regulator')
plt.title('ACSC Projection Φ(E)')
plt.colorbar(sc, label='Rank')
plt.tight_layout()
plt.savefig('results/projection_scatter.png', dpi=200)
plt.show()
print("Saved results/projection_scatter.png")


In [ ]:
try:
    from ripser import ripser
    from persim import plot_diagrams
    D = np.linalg.norm(points[:,None,:] - points[None,:,:], axis=2)
    diagrams = ripser(D, distance_matrix=True, maxdim=1)['dgms']
    plot_diagrams(diagrams, show=True)
    print("Computed persistence diagrams (ripser).")
except Exception as e:
    print("ripser/persim not available or failed:", e)


**Interpretation.** The scatter shows the embedding of elliptic-curve invariants into a 3D symbolic manifold. Next: compute entropy on these points (Notebook 02) and analyze topology (Notebook 07).


In [ ]:
print("Notebook: 01_projection_demo")
print("Data used:", data_path if data_path else "synthetic")
print("Outputs: results/projection_points.csv, results/projection_scatter.png")
